# Pro Submission — Stacked Ensemble With Compositional Features

Goal: push past the current leaderboard top (`0.7542`) by targeting the weakest labels (`distance_bin`, `followup_protocol`, `precursor_category`).

Key additions vs `solution_final`:
1. Char 3-5 and word 1-2 TF-IDF features to capture both lexical and morphological signals.
2. Bucketed redshift and log-luminosity features (quantile bins learned from training data).
3. Compositional features: outer product of out-of-fold class and environment probabilities, which preserves information about unseen `(class, environment)` pairs without binding them to training joint counts.
4. 10-fold OOF stage-1 probabilities for class / environment / regime / variability.
5. Stage-2 blend per noisy label: logistic regression on the full sparse stack + 3-seed HistGradientBoosting ensemble on the dense view.
6. Defensive validation so every prediction is a valid value for its column.

In [1]:
import os
import re
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import OneHotEncoder, StandardScaler

DATA_DIR = './dataset/public'
OUTPUT_PATH = './working/submission.csv'
os.makedirs('./working', exist_ok=True)

train = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
test = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))

LABEL_COLUMNS = [
    'transient_class', 'host_environment', 'spectral_regime',
    'variability_pattern', 'distance_bin', 'energy_tier',
    'followup_protocol', 'precursor_category',
]
CLEAN_LABELS = LABEL_COLUMNS[:3]
NOISY_LABELS = LABEL_COLUMNS[3:]
STAGE1_LABELS = CLEAN_LABELS + ['variability_pattern']
VALID_VALUES = {c: sorted(train[c].dropna().unique().tolist()) for c in LABEL_COLUMNS}

In [2]:
_Z = re.compile(r'z\s*=\s*(\d+(?:\.\d+)?)', re.IGNORECASE)
_L = re.compile(r'log\s*L\s*=\s*(\d+(?:\.\d+)?)', re.IGNORECASE)

vocab_caches = {
    col: [(c, c.lower().replace('_', ' ')) for c in VALID_VALUES[col]]
    for col in STAGE1_LABELS
}

def lex(text, voc):
    if not isinstance(text, str):
        return '__none__'
    lowered = text.lower()
    best = '__none__'
    best_pos = len(lowered) + 1
    for cand, needle in voc:
        pos = lowered.find(needle)
        if pos != -1 and pos < best_pos:
            best = cand
            best_pos = pos
    return best

def build_struct(df):
    z = df['narrative'].apply(lambda t: _Z.search(t).group(1) if isinstance(t, str) and _Z.search(t) else None)
    L = df['narrative'].apply(lambda t: _L.search(t).group(1) if isinstance(t, str) and _L.search(t) else None)
    out = pd.DataFrame({'id': df['id'].astype(str).values})
    out['z_value'] = pd.to_numeric(z, errors='coerce').astype(float)
    out['log_L'] = pd.to_numeric(L, errors='coerce').astype(float)
    out['z_missing'] = z.isna().astype(float)
    out['L_missing'] = L.isna().astype(float)
    for col in STAGE1_LABELS:
        out[f'{col}_lex'] = df['narrative'].apply(lambda t, v=vocab_caches[col]: lex(t, v))
    return out

train_s = build_struct(train)
test_s = build_struct(test)

In [3]:
# Learn z / L quantile bins from training rows with a valid numeric value and
# expose one-hot encoded bin IDs for both train and test. Rows with missing
# values fall into a dedicated 'missing' bin.
z_valid = train_s.loc[train_s['z_missing'] == 0, 'z_value'].values
L_valid = train_s.loc[train_s['L_missing'] == 0, 'log_L'].values

z_edges = np.quantile(z_valid, np.linspace(0, 1, 9))
L_edges = np.quantile(L_valid, np.linspace(0, 1, 7))

def bucketize(df_s, edges, value_col, missing_col, label):
    values = df_s[value_col].values
    missing = df_s[missing_col].values.astype(bool)
    bins = np.digitize(np.nan_to_num(values), edges[1:-1], right=True)
    bins = bins.astype(int)
    bins[missing] = len(edges)  # reserve the last index for missing
    return pd.Categorical([f'{label}_{b}' for b in bins])

train_s['z_bucket'] = bucketize(train_s, z_edges, 'z_value', 'z_missing', 'z')
test_s['z_bucket'] = bucketize(test_s, z_edges, 'z_value', 'z_missing', 'z')
train_s['L_bucket'] = bucketize(train_s, L_edges, 'log_L', 'L_missing', 'L')
test_s['L_bucket'] = bucketize(test_s, L_edges, 'log_L', 'L_missing', 'L')

In [4]:
word_vec = TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_df=0.95, sublinear_tf=True, max_features=15000)
char_vec = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 4), min_df=3, max_df=0.95, sublinear_tf=True, max_features=15000)
word_vec.fit(train['narrative'])
char_vec.fit(train['narrative'])

X_word_train = word_vec.transform(train['narrative'])
X_word_test = word_vec.transform(test['narrative'])
X_char_train = char_vec.transform(train['narrative'])
X_char_test = char_vec.transform(test['narrative'])

cat_cols = [f'{c}_lex' for c in STAGE1_LABELS] + ['z_bucket', 'L_bucket']
ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=True)
ohe.fit(pd.concat([train_s[cat_cols].astype(str), test_s[cat_cols].astype(str)], ignore_index=True))
X_cat_train = ohe.transform(train_s[cat_cols].astype(str))
X_cat_test = ohe.transform(test_s[cat_cols].astype(str))

scaler = StandardScaler().fit(train_s[['z_value', 'log_L']].fillna(0.0).values)
num_vals_train = scaler.transform(train_s[['z_value', 'log_L']].fillna(0.0).values) * 0.1
num_vals_test = scaler.transform(test_s[['z_value', 'log_L']].fillna(0.0).values) * 0.1
num_flags_train = train_s[['z_missing', 'L_missing']].values.astype(np.float32)
num_flags_test = test_s[['z_missing', 'L_missing']].values.astype(np.float32)
X_num_train = sparse.csr_matrix(np.concatenate([num_vals_train.astype(np.float32), num_flags_train], axis=1))
X_num_test = sparse.csr_matrix(np.concatenate([num_vals_test.astype(np.float32), num_flags_test], axis=1))

X_text_only_train = sparse.hstack([X_word_train, X_char_train]).tocsr()
X_text_only_test = sparse.hstack([X_word_test, X_char_test]).tocsr()
X_full_train = sparse.hstack([X_word_train, X_char_train, X_cat_train, X_num_train]).tocsr()
X_full_test = sparse.hstack([X_word_test, X_char_test, X_cat_test, X_num_test]).tocsr()
print('text features', X_text_only_train.shape, 'full features', X_full_train.shape)

text features (7470, 27033) full features (7470, 27086)


In [5]:
def align_proba(probs, clf_classes, ordered_classes):
    aligned = np.zeros((probs.shape[0], len(ordered_classes)), dtype=np.float32)
    for idx, cls in enumerate(clf_classes):
        aligned[:, list(ordered_classes).index(cls)] = probs[:, idx]
    return aligned

oof_train_probs = {}
test_probs = {}
final_predictions = {'id': test['id'].astype(str).values}

skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=0)
for col in STAGE1_LABELS:
    classes = np.array(VALID_VALUES[col])
    oof = np.zeros((X_text_only_train.shape[0], len(classes)), dtype=np.float32)
    y = train[col].values
    for fold, (tr, va) in enumerate(skf.split(X_text_only_train, y)):
        clf = LogisticRegression(max_iter=400, solver='lbfgs')
        clf.fit(X_text_only_train[tr], y[tr])
        oof[va] = align_proba(clf.predict_proba(X_text_only_train[va]), clf.classes_, classes)
    oof_train_probs[col] = oof
    final_clf = LogisticRegression(max_iter=600, solver='lbfgs')
    final_clf.fit(X_text_only_train, y)
    test_probs[col] = align_proba(final_clf.predict_proba(X_text_only_test), final_clf.classes_, classes)
    if col in CLEAN_LABELS:
        lex_col = test_s[f'{col}_lex'].values
        model_pred = final_clf.predict(X_text_only_test)
        final_predictions[col] = np.where(
            (lex_col != '__none__') & np.isin(lex_col, classes),
            lex_col, model_pred,
        )
    print(f'stage1 {col:22s} done')

stage1 transient_class        done


stage1 host_environment       done


stage1 spectral_regime        done


stage1 variability_pattern    done


In [6]:
# Build the dense view for the HistGradientBoosting ensemble. Adds a compositional
# outer product of class and environment probabilities so models can factorize
# even for unseen (class, environment) combinations.
class_tr = oof_train_probs['transient_class']
env_tr = oof_train_probs['host_environment']
regime_tr = oof_train_probs['spectral_regime']
var_tr = oof_train_probs['variability_pattern']
class_te = test_probs['transient_class']
env_te = test_probs['host_environment']
regime_te = test_probs['spectral_regime']
var_te = test_probs['variability_pattern']

comp_tr = (class_tr[:, :, None] * env_tr[:, None, :]).reshape(class_tr.shape[0], -1)
comp_te = (class_te[:, :, None] * env_te[:, None, :]).reshape(class_te.shape[0], -1)

num_view_train = np.concatenate([
    train_s[['z_value', 'log_L']].fillna(0.0).values.astype(np.float32),
    train_s[['z_missing', 'L_missing']].values.astype(np.float32),
], axis=1)
num_view_test = np.concatenate([
    test_s[['z_value', 'log_L']].fillna(0.0).values.astype(np.float32),
    test_s[['z_missing', 'L_missing']].values.astype(np.float32),
], axis=1)

X_dense_train = np.concatenate([class_tr, env_tr, regime_tr, var_tr, comp_tr, num_view_train], axis=1)
X_dense_test = np.concatenate([class_te, env_te, regime_te, var_te, comp_te, num_view_test], axis=1)
print('dense shape', X_dense_train.shape)

dense shape (7470, 144)


In [7]:
for col in NOISY_LABELS:
    classes = np.array(VALID_VALUES[col])
    log_clf = LogisticRegression(max_iter=400, C=1.0, solver='lbfgs')
    log_clf.fit(X_full_train, train[col].values)
    log_probs = align_proba(log_clf.predict_proba(X_full_test), log_clf.classes_, classes)

    gb_probs_sum = np.zeros_like(log_probs)
    for seed in (0, 1, 2):
        gb_clf = HistGradientBoostingClassifier(
            max_iter=300,
            learning_rate=0.06,
            max_leaf_nodes=31,
            l2_regularization=0.5,
            random_state=seed,
        )
        gb_clf.fit(X_dense_train, train[col].values)
        gb_probs_sum += align_proba(gb_clf.predict_proba(X_dense_test), gb_clf.classes_, classes)
    gb_probs = gb_probs_sum / 3.0

    blended = 0.55 * log_probs + 0.45 * gb_probs
    final_predictions[col] = classes[np.argmax(blended, axis=1)]
    print(f'stage2 {col:22s} done')

submission = pd.DataFrame(final_predictions)[['id'] + LABEL_COLUMNS]
for col in LABEL_COLUMNS:
    valid = set(VALID_VALUES[col])
    fallback = train[col].mode().iloc[0]
    submission[col] = submission[col].where(submission[col].isin(valid), fallback)
submission.to_csv(OUTPUT_PATH, index=False)
print('wrote', OUTPUT_PATH, submission.shape)
submission.head()

stage2 variability_pattern    done


stage2 distance_bin           done


stage2 energy_tier            done


stage2 followup_protocol      done


stage2 precursor_category     done
wrote ./working/submission.csv (2520, 9)


,id,transient_class,host_environment,spectral_regime,variability_pattern,distance_bin,energy_tier,followup_protocol,precursor_category
0,7471,Hyperaccretion Flare,Galactic Bar Vicinity,uv,chaotic,mid,medium_high,PROTOCOL_D,CAT_2
1,7472,Tidally Locked Beacon,Circumnuclear Disk,infrared,quasi_periodic,near,low,PROTOCOL_E,CAT_5
2,7473,Hyperaccretion Flare,AGN Wind Region,uv,monotonic_rise,mid_far,medium_high,PROTOCOL_C,CAT_2
3,7474,Hot Jet Eruption,Stellar Stream,soft_xray,monotonic_rise,far,high,PROTOCOL_C,CAT_6
4,7475,Hot Jet Eruption,Circumnuclear Disk,radio,monotonic_rise,mid_far,medium,PROTOCOL_F,CAT_2
